In [0]:
%sql
select *, array_join(hotwords, ' ') as hotwords from nasa_eonet.events_gold.dim_categories;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql 
select * from 
nasa_eonet.events_gold.fact_events
left join
nasa_eonet.events_gold.dim_points
on 

In [0]:
%sql
select name, count(*) from 
nasa_eonet.events_gold.fact_events as e
left join
nasa_eonet.events_gold.dim_points as p
on 
e.`point_id` = p.`point_id`
group by name

-- for the chorpleth, the name column doesnt exactly match databrick's internal names

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
select date_format(min_date, 'MMMM') as month, month(min_date) as month_num, count(*) as event_count
from 
nasa_eonet.events_gold.fact_events e
group by date_format(min_date, 'MMMM'), month(min_date)
order by month_num

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
select count(e.id) as count_of_events, c.id as c_id, c.title as c_title
from (
  select *
  from nasa_eonet.events_gold.fact_events as e
  lateral view explode(e.category_id) cats as exploded_category
) as e
join 
nasa_eonet.events_gold.dim_categories as c
on
e.exploded_category = c.id
group by c_id, c_title

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
select 
fe.id, fe.event_duration, fe.min_date, fe.name,
ds.id as source_id, ds.title as source_title,
dc.id as category_id, dc.title as category_title
from (
  select *
  from nasa_eonet.events_gold.fact_events as e
  lateral view explode(e.category_id) cats as exploded_category
  lateral view explode(e.source_id) srcs as exploded_source
) as fe
join 
nasa_eonet.events_gold.dim_sources as ds
on
fe.exploded_source = ds.id
join 
nasa_eonet.events_gold.dim_categories as dc
on
fe.exploded_category = dc.id
limit 10;

In [0]:

%sql
select 
  e.id, e.title, e.category_id, e.name, e.`ISO3166-1-Alpha-3`, 
  p.point_id,
  st_x(p.point_geometry) as lon, st_y(p.point_geometry) as lat
from (
  select *
  from nasa_eonet.events_gold.fact_events
  lateral view explode(category_id) cats as exploded_category
) as e
join
nasa_eonet.events_gold.dim_points p 
on
e.point_id = p.point_id 


In [0]:

%sql
-- Isolate the point_id mismatch between fact_events and dim_points
select 'events_with_null_point_id' as check_name, count(*) as cnt
from nasa_eonet.events_gold.fact_events
where point_id is null

union all

select 'duplicate_point_ids_in_dim_points', count(*)
from (
  select point_id
  from nasa_eonet.events_gold.dim_points
  group by point_id
  having count(*) > 1
)

union all

select 'event_point_ids_missing_from_dim_points', count(*)
from (
  select distinct point_id
  from nasa_eonet.events_gold.fact_events
  except
  select distinct point_id
  from nasa_eonet.events_gold.dim_points
)

union all

select 'dim_point_ids_not_used_by_events', count(*)
from (
  select distinct point_id
  from nasa_eonet.events_gold.dim_points
  except
  select distinct point_id
  from nasa_eonet.events_gold.fact_events
)

-- Then drill into the offending values, e.g.:
-- select e.point_id, count(*) as event_count
-- from nasa_eonet.events_gold.fact_events e
-- left join nasa_eonet.events_gold.dim_points p
--   on e.point_id = p.point_id
-- where p.point_id is null
-- group by e.point_id

In [0]:
%sql
select count(*) from nasa_eonet.events_gold.dim_points